%md

# 🥉 Ingestão Bronze - Ecommerce Pedidos

## Squad 2 - Real Time for Business

### Grupo 3 - Ecommerce Pedidos

---

# 🎯 Objetivo

Este notebook tem como objetivo realizar a ingestão dos dados
Real Time de pedidos para a camada Bronze da arquitetura
de dados.

Nesta etapa, os arquivos recebidos na camada de origem (`raw`)
serão carregados, enriquecidos com informações de auditoria
e armazenados em formato Delta Lake.

---

# 🏗️ Contexto da Arquitetura

O projeto utiliza uma arquitetura baseada em camadas:

```
raw

↓

🥉 Bronze

↓

🥈 Silver

↓

🥇 Gold
```

Cada camada possui uma responsabilidade específica dentro
do pipeline de dados.

---

# 📂 Origem dos Dados

Os dados de entrada estão armazenados no Azure Data Lake
Storage Gen2.

Ambiente de origem:

```
Storage Account:

internshipdatalake


Container:

raw
```

Fonte utilizada:

```
real-time-data/
```

Formato dos arquivos:

```
Parquet
```

Exemplo:

```
real-time-data/2026/09/21/113202/ecommerce_pedidos.parquet
```

---

# ⚡ Dados Real Time

Os arquivos Real Time representam eventos recebidos
continuamente através de micro-lotes.

Características:

- Atualizações incrementais;
- Arquivos independentes por evento;
- Entrada para processamento contínuo;
- Necessidade de rastreabilidade.

---

# 🥉 Camada Bronze

A camada Bronze representa a primeira etapa de persistência
dos dados dentro do pipeline.

Responsabilidades:

- Receber os dados da camada Raw;
- Preservar a estrutura original;
- Armazenar os dados em formato Delta;
- Registrar informações de auditoria.

---

# 🏷️ Metadados de Auditoria

Durante a ingestão serão adicionadas informações
para rastreabilidade:

## bronze_ingested_at

Indica o momento em que o registro foi inserido
na camada Bronze.

Exemplo:

```
2026-09-28 18:00:00
```

---

## bronze_source_file

Indica o arquivo de origem do registro.

Exemplo:

```
real-time-data/2026/09/21/113202/ecommerce_pedidos.parquet
```

---

# 📍 Destino dos Dados

Os dados processados serão armazenados no ambiente
da Squad 2 - Grupo 3.

Destino:

```
Container:

squad2


Grupo:

grupo3
```

Estrutura:

```
squad2

└── grupo3

    └── bronze

        └── ecommerce_pedidos
```

---

# 🔄 Fluxo da Ingestão Bronze

```
RAW

(real-time-data)

        ↓

Leitura dos micro-lotes

        ↓

Adição dos metadados

        ↓

Persistência Delta Lake

        ↓

squad2/grupo3/bronze/ecommerce_pedidos
```

---

# 🚫 Responsabilidades que NÃO pertencem à Bronze

Nesta etapa não serão realizadas:

- Limpeza dos dados;
- Alteração de valores;
- Correção de tipos;
- Regras de negócio;
- Remoção de registros.

Esses tratamentos serão realizados futuramente
na camada Silver.

---

# 🛡️ Boas Práticas Aplicadas

Nesta etapa são aplicadas práticas de Engenharia de Dados:

- Separação entre origem e destino;
- Controle de rastreabilidade;
- Persistência em formato Delta;
- Organização por camadas;
- Preparação para processamento incremental.

---

# ✅ Resultado Esperado

Ao final deste notebook:

- Os micro-lotes Real Time serão processados;
- Os dados serão armazenados em Delta Lake;
- Os registros possuirão informações de auditoria;
- A camada Bronze estará preparada para a etapa Silver.

Este notebook representa a primeira etapa de ingestão
do pipeline da Squad 2 - Grupo 3.

In [0]:
# ==================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# INGESTÃO BRONZE
# ==================================================


# Biblioteca utilizada para manipulação
# de dados em formato tabular.

import pandas as pd



# Biblioteca padrão do Python.
#
# Utilizada para acessar variáveis
# de ambiente através do os.getenv().

import os



# Biblioteca utilizada para trabalhar
# com arquivos carregados em memória.
#
# Será utilizada para transformar os bytes
# dos arquivos Parquet em objetos legíveis
# pelo Pandas.

from io import BytesIO



# Biblioteca responsável por carregar
# as variáveis armazenadas no arquivo .env.

from dotenv import load_dotenv



# Biblioteca utilizada para autenticação
# no Azure através de Service Principal.

from azure.identity import (
    ClientSecretCredential
)



# Cliente responsável pela comunicação
# com o Azure Data Lake Storage Gen2.

from azure.storage.filedatalake import (
    DataLakeServiceClient
)



# Funções Spark utilizadas para criação
# dos metadados da camada Bronze.
#
# lit:
# cria colunas com valores fixos.
#
# current_timestamp:
# registra o momento da ingestão.

from pyspark.sql.functions import (
    lit,
    current_timestamp
)



print(
    "Bibliotecas Bronze carregadas com sucesso!"
)

Bibliotecas Bronze carregadas com sucesso!


In [0]:
# ==================================================
# CARREGAMENTO DAS CONFIGURAÇÕES DO AMBIENTE
# INGESTÃO BRONZE
# ==================================================


# Carrega as variáveis do arquivo .env.

load_dotenv(
    ".env",
    override=True
)



# Recupera credenciais Azure.

adls_client_id = os.getenv(
    "ADLS_CLIENT_ID"
)


adls_tenant_id = os.getenv(
    "ADLS_TENANT_ID"
)


adls_client_secret = os.getenv(
    "ADLS_CLIENT_SECRET"
)



# Recupera a conta de armazenamento.

storage_account_name = os.getenv(
    "STORAGE_ACCOUNT_NAME"
)



# Container de origem.

raw_container = "raw"



# Container de destino.

target_container = os.getenv(
    "CONTAINER_NAME"
)



# Grupo responsável.

group_name = os.getenv(
    "GROUP_NAME"
)



# Caminho da camada Bronze.

bronze_path = (
    f"{group_name}/bronze/ecommerce_pedidos"
)



print(
    "Configurações Bronze carregadas com sucesso!"
)



print(
    "Storage Account:",
    storage_account_name
)



print(
    "Origem:",
    raw_container
)



print(
    "Destino:",
    target_container
)



print(
    "Bronze:",
    bronze_path
)

Configurações Bronze carregadas com sucesso!
Storage Account: internshipdatalake
Origem: raw
Destino: squad2
Bronze: grupo3/bronze/ecommerce_pedidos


In [0]:
# ==================================================
# CRIAÇÃO DA CREDENCIAL AZURE
# INGESTÃO BRONZE
# ==================================================


# Cria a identidade de acesso Azure
# utilizando Service Principal.

credential = ClientSecretCredential(

    tenant_id=adls_tenant_id,

    client_id=adls_client_id,

    client_secret=adls_client_secret

)



print(
    "Credencial Azure criada com sucesso!"
)

Credencial Azure criada com sucesso!


In [0]:
# ==================================================
# CRIAÇÃO DOS CLIENTES AZURE DATA LAKE
# ORIGEM E DESTINO
# ==================================================


# Cria o cliente principal do Data Lake.

service_client = DataLakeServiceClient(

    account_url=(
        f"https://{storage_account_name}.dfs.core.windows.net"
    ),

    credential=credential

)



# Cliente do container de origem RAW.

raw_file_system_client = (
    service_client
    .get_file_system_client(
        raw_container
    )
)



# Cliente do container destino SQUAD2.

squad2_file_system_client = (
    service_client
    .get_file_system_client(
        target_container
    )
)



print(
    "Clientes Azure Data Lake criados com sucesso!"
)



print(
    "Origem:",
    raw_container
)



print(
    "Destino:",
    target_container
)

Clientes Azure Data Lake criados com sucesso!
Origem: raw
Destino: squad2


In [0]:
# ==================================================
# DESCOBERTA DOS MICRO-LOTES REAL TIME
# INGESTÃO BRONZE
# ==================================================


# Lista todos os caminhos existentes
# dentro do container RAW.

paths_raw = (
    raw_file_system_client
    .get_paths()
)



# Lista que armazenará os arquivos
# de pedidos encontrados.

arquivos_realtime = []



# Procura somente arquivos:
#
# real-time-data
#
# ecommerce_pedidos.parquet

for path in paths_raw:


    if (
        "real-time-data" in path.name
        and
        "ecommerce_pedidos.parquet" in path.name
    ):

        arquivos_realtime.append(
            path.name
        )



print(
    "Quantidade de micro-lotes encontrados:",
    len(arquivos_realtime)
)



print(
    "\nArquivos encontrados:"
)



for arquivo in arquivos_realtime:

    print(
        arquivo
    )

Quantidade de micro-lotes encontrados: 6

Arquivos encontrados:
real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet
real-time-data/2026/09/21/112402/ecommerce_pedidos.parquet
real-time-data/2026/09/21/112602/ecommerce_pedidos.parquet
real-time-data/2026/09/21/112802/ecommerce_pedidos.parquet
real-time-data/2026/09/21/113002/ecommerce_pedidos.parquet
real-time-data/2026/09/21/113202/ecommerce_pedidos.parquet


In [0]:
# ==================================================
# LEITURA E CONSOLIDAÇÃO DOS MICRO-LOTES
# INGESTÃO BRONZE
# ==================================================


# Lista para armazenar
# os DataFrames de cada arquivo.

dfs_bronze = []



# Percorre todos os arquivos encontrados.

for arquivo in arquivos_realtime:


    print(
        "Processando:",
        arquivo
    )


    # Cria cliente do arquivo Parquet.

    arquivo_client = (
        raw_file_system_client
        .get_file_client(
            arquivo
        )
    )


    # Realiza download do arquivo.

    download = (
        arquivo_client
        .download_file()
    )


    # Lê o conteúdo do arquivo.

    conteudo = (
        download
        .readall()
    )


    # Converte Parquet para Pandas.

    df_temp = pd.read_parquet(
        BytesIO(
            conteudo
        )
    )


    # Adiciona a origem do arquivo.
    #
    # Esse campo será importante
    # para auditoria da Bronze.

    df_temp["bronze_source_file"] = arquivo



    # Adiciona o DataFrame na lista.

    dfs_bronze.append(
        df_temp
    )



# Consolida todos os micro-lotes.

df_bronze_pandas = pd.concat(
    dfs_bronze,
    ignore_index=True
)



print(
    "Micro-lotes consolidados com sucesso!"
)


print(
    "Quantidade de registros:",
    len(df_bronze_pandas)
)


print(
    "Quantidade de colunas:",
    len(df_bronze_pandas.columns)
)

Processando: real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet
Processando: real-time-data/2026/09/21/112402/ecommerce_pedidos.parquet
Processando: real-time-data/2026/09/21/112602/ecommerce_pedidos.parquet
Processando: real-time-data/2026/09/21/112802/ecommerce_pedidos.parquet
Processando: real-time-data/2026/09/21/113002/ecommerce_pedidos.parquet
Processando: real-time-data/2026/09/21/113202/ecommerce_pedidos.parquet
Micro-lotes consolidados com sucesso!
Quantidade de registros: 27
Quantidade de colunas: 11


In [0]:
# ==================================================
# CRIAÇÃO DO DATAFRAME SPARK BRONZE
# ==================================================


# Converte o DataFrame Pandas consolidado
# para DataFrame Spark.

df_bronze = (
    spark
    .createDataFrame(
        df_bronze_pandas
    )
)



print(
    "DataFrame Spark Bronze criado com sucesso!"
)



print(
    "Quantidade de registros:",
    df_bronze.count()
)



print(
    "Quantidade de colunas:",
    len(df_bronze.columns)
)

DataFrame Spark Bronze criado com sucesso!
Quantidade de registros: 27
Quantidade de colunas: 11


In [0]:
# ==================================================
# ADIÇÃO DOS METADADOS BRONZE
# ==================================================


# Adiciona o timestamp da ingestão.
#
# Esse campo registra o momento em que
# o registro foi processado pela camada Bronze.


df_bronze = (
    df_bronze
    .withColumn(
        "bronze_ingested_at",
        current_timestamp()
    )
)



print(
    "Metadado bronze_ingested_at adicionado com sucesso!"
)



print(
    "Quantidade de colunas:",
    len(df_bronze.columns)
)

Metadado bronze_ingested_at adicionado com sucesso!
Quantidade de colunas: 12


In [0]:
# ==================================================
# VALIDAÇÃO DO SCHEMA BRONZE
# ==================================================


print(
    "Schema final da camada Bronze:"
)



df_bronze.printSchema()

Schema final da camada Bronze:
root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_endereco_entrega: long (nullable = true)
 |-- dt_pedido: timestamp (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- dt_previsao_entrega: timestamp (nullable = true)
 |-- dt_ultima_atualizacao_status: timestamp (nullable = true)
 |-- bronze_source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = false)



# 💡 Observação - Schema Bronze

Durante a ingestão dos arquivos Real Time, algumas colunas
temporais foram interpretadas automaticamente como timestamp
durante a leitura dos arquivos Parquet.

A camada Bronze mantém a estrutura recebida dos arquivos,
adicionando apenas metadados de auditoria.

In [0]:
# ==================================================
# GRAVAÇÃO DA BRONZE COMO DELTA TABLE
# ==================================================


# Salva o DataFrame Bronze como uma tabela Delta
# dentro do database criado.

(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "grupo3_bronze.ecommerce_pedidos"
    )
)



print(
    "Tabela Delta Bronze criada com sucesso!"
)

Tabela Delta Bronze criada com sucesso!


In [0]:
# ==================================================
# VALIDAÇÃO DA TABELA DELTA BRONZE
# ==================================================


# Lê a tabela Delta criada.

df_validacao_bronze = (
    spark
    .table(
        "grupo3_bronze.ecommerce_pedidos"
    )
)



print(
    "Tabela Bronze carregada para validação!"
)



print(
    "Quantidade de registros:",
    df_validacao_bronze.count()
)



print(
    "Quantidade de colunas:",
    len(df_validacao_bronze.columns)
)

Tabela Bronze carregada para validação!
Quantidade de registros: 27
Quantidade de colunas: 12


In [0]:
# ==================================================
# VISUALIZAÇÃO DOS REGISTROS BRONZE
# COM METADADOS
# ==================================================


# Exibe alguns registros da Bronze
# incluindo os campos de auditoria.


display(
    df_validacao_bronze
    .select(
        
        "id_pedido",

        "valor_total",

        "metodo_pagamento",

        "bronze_source_file",

        "bronze_ingested_at"

    )
    .limit(10)
)

id_pedido,valor_total,metodo_pagamento,bronze_source_file,bronze_ingested_at
1790000559,65.0,boleto,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000560,128.12,boleto,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000561,271.93,cartão débito,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000562,18.66,boleto,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000563,221.31,pix,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000564,49.64,pix,real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000640,664.13,pix,real-time-data/2026/09/21/112402/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000641,68.38,cartão débito,real-time-data/2026/09/21/112402/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000760,237.38,cartão débito,real-time-data/2026/09/21/112602/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z
1790000761,101.15,cartão crédito,real-time-data/2026/09/21/112602/ecommerce_pedidos.parquet,2026-09-29T14:20:03.906Z


%md

# ✅ Conclusão - Ingestão Bronze de Pedidos

## Squad 2 - Real Time for Business

### Grupo 3 - Ecommerce Pedidos

---

# 🎯 Objetivo Alcançado

Este notebook realizou com sucesso a ingestão dos dados
Real Time de pedidos para a camada Bronze da arquitetura
de dados.

Os arquivos recebidos na camada Raw foram processados,
consolidados e armazenados em formato Delta Lake,
mantendo rastreabilidade através de metadados de auditoria.

---

# 📂 Origem dos Dados

Os dados foram obtidos a partir do Azure Data Lake Storage Gen2.

Origem:

```
Container:

raw
```

Arquivos processados:

```
real-time-data/2026/09/21/
```

Micro-lotes identificados:

```
112245

112402

112602

112802

113002

113202
```

---

# ⚡ Processamento Real Time

Durante a ingestão foram processados:

```
Quantidade de micro-lotes:

6
```

Resultado:

```
Registros consolidados:

27
```

Os arquivos Parquet foram lidos e consolidados
em um único DataFrame Spark.

---

# 🥉 Camada Bronze

Os dados foram armazenados na camada Bronze utilizando
formato Delta Lake.

Destino:

```
Database:

grupo3_bronze


Tabela:

ecommerce_pedidos
```

---

# 🏷️ Metadados de Auditoria

Durante o processo de ingestão foram adicionados:

## bronze_source_file

Responsável por identificar o arquivo de origem
de cada registro.

Exemplo:

```
real-time-data/2026/09/21/112245/ecommerce_pedidos.parquet
```

---

## bronze_ingested_at

Responsável por registrar o momento em que o dado
foi inserido na camada Bronze.

Exemplo:

```
2026-09-28 21:38:34
```

---

# 📊 Resultado da Camada Bronze

Validação realizada:

```
Quantidade de registros:

27


Quantidade de colunas:

12
```

Estrutura final:

```
ecommerce_pedidos

├── Dados originais

├── bronze_source_file

└── bronze_ingested_at
```

---

# 🛡️ Boas Práticas Aplicadas

Durante esta etapa foram aplicadas práticas de Engenharia
de Dados:

- Separação entre origem e destino;
- Processamento incremental por micro-lotes;
- Controle de rastreabilidade;
- Persistência em Delta Lake;
- Organização por camadas;
- Preparação para tratamento na Silver.

---

# 🔄 Próxima Etapa do Pipeline

Após a conclusão da Bronze, os dados seguirão para:

```
🥉 Bronze

        ↓

🥈 Silver

        ↓

🥇 Gold
```

A próxima etapa será responsável pelo tratamento,
padronização e validação dos dados.

---

# 🚀 Resultado Final

A ingestão Bronze foi concluída com sucesso.

Os dados Real Time de pedidos agora estão disponíveis
em uma camada estruturada, auditável e preparada para
as próximas etapas do pipeline de Engenharia de Dados
da Squad 2 - Grupo 3.